# LAB 4 — Modular RAG: Pipeline com Módulos Intercambiáveis
## MBA RAG & CAG Aplicados a Direito e Segurança Pública — Aula 3

**Objetivo:** Construir um pipeline Modular RAG onde o retriever pode ser trocado
entre OpenSearch e ChromaDB sem alterar nenhum outro módulo.

**Entregáveis:**
- Pipeline Modular RAG com interface abstrata funcional
- Demonstração de troca de retriever OpenSearch ↔ ChromaDB
- Análise: o resultado mudou com retrievers diferentes?

**Tempo estimado:** 40 minutos  
**Conceito central:** Interfaces abstratas (ABC) + injeção de dependência


## ⚙️ Passo 1 — Instalação

In [5]:
%pip install langchain langchain-community openai opensearch-py chromadb sentence-transformers
print("✅ Instalação concluída")

   ---------------------------------------- 0.0/23.5 MB ? eta -:--:--
   --- ------------------------------------ 1.8/23.5 MB 12.6 MB/s eta 0:00:02
   ---- ----------------------------------- 2.9/23.5 MB 7.3 MB/s eta 0:00:03
   ------- -------------------------------- 4.2/23.5 MB 7.2 MB/s eta 0:00:03
   -------- ------------------------------- 5.2/23.5 MB 6.8 MB/s eta 0:00:03
   ----------- ---------------------------- 6.8/23.5 MB 6.9 MB/s eta 0:00:03
   -------------- ------------------------- 8.4/23.5 MB 7.0 MB/s eta 0:00:03
   ---------------- ----------------------- 10.0/23.5 MB 7.0 MB/s eta 0:00:02
   -------------------- ------------------- 11.8/23.5 MB 7.2 MB/s eta 0:00:02
   ---------------------- ----------------- 13.1/23.5 MB 7.1 MB/s eta 0:00:02
   ------------------------ --------------- 14.7/23.5 MB 7.2 MB/s eta 0:00:02
   --------------------------- ------------ 16.3/23.5 MB 7.2 MB/s eta 0:00:02
   ------------------------------ --------- 17.8/23.5 MB 7.3 MB/s eta 0:00:01

## 🏗️ Passo 2 — Definir as Interfaces Abstratas

In [6]:
from abc import ABC, abstractmethod
from typing import List, Dict, Optional
from dataclasses import dataclass

@dataclass
class Document:
    """Representação padronizada de um documento recuperado."""
    id: str
    content: str
    score: float
    metadata: Dict = None
    
    def __post_init__(self):
        if self.metadata is None:
            self.metadata = {}

class BaseRetriever(ABC):
    """Interface contratual para todos os retrievers.
    
    Qualquer retriever no pipeline DEVE implementar este contrato.
    Isso garante intercambialidade sem alterar o resto do pipeline.
    """
    
    @abstractmethod
    def search(self, query: str, top_k: int = 10) -> List[Document]:
        """
        Busca documentos relevantes para a query.
        
        Args:
            query: Query de busca
            top_k: Número máximo de resultados
            
        Returns:
            Lista de Documents ordenados por relevância (maior score primeiro)
        """
        pass
    
    @abstractmethod
    def add_documents(self, documents: List[Dict]) -> None:
        """Indexa novos documentos."""
        pass
    
    @property
    @abstractmethod
    def name(self) -> str:
        """Nome identificador do retriever."""
        pass

class BaseReranker(ABC):
    @abstractmethod
    def rerank(self, query: str, docs: List[Document], top_k: int = 5) -> List[Document]:
        pass

class BaseGenerator(ABC):
    @abstractmethod
    def generate(self, query: str, context: str) -> str:
        pass

print("✅ Interfaces abstratas definidas:")
print("   BaseRetriever  — search(), add_documents(), name")
print("   BaseReranker   — rerank()")
print("   BaseGenerator  — generate()")

✅ Interfaces abstratas definidas:
   BaseRetriever  — search(), add_documents(), name
   BaseReranker   — rerank()
   BaseGenerator  — generate()


## 🔌 Passo 3 — Implementar Retriever OpenSearch

In [7]:
import os
from opensearchpy import OpenSearch

class OpenSearchRetriever(BaseRetriever):
    """Retriever usando OpenSearch BM25 + kNN."""
    
    def __init__(self, host: str = "localhost", port: int = 9200,
                 index: str = "corpus_juridico", password: str = "admin"):
        self._name = "OpenSearch"
        self.index = index
        try:
            self.client = OpenSearch(
                hosts=[{"host": host, "port": port}],
                http_auth=("admin", password),
                use_ssl=False, verify_certs=False
            )
            self.client.info()
            self.connected = True
            print(f"✅ OpenSearchRetriever conectado ao índice '{index}'")
        except Exception as e:
            self.connected = False
            print(f"⚠️  OpenSearch não disponível ({e}) — modo simulado")
    
    @property
    def name(self) -> str:
        return self._name
    
    def search(self, query: str, top_k: int = 10) -> List[Document]:
        if not self.connected:
            return self._simulado(query, top_k)
        
        resp = self.client.search(
            index=self.index,
            body={"query": {"multi_match": {"query": query,
                   "fields": ["texto^2", "ementa^1.5", "palavras_chave"]}},
                  "size": top_k}
        )
        return [
            Document(
                id=h["_id"],
                content=h["_source"].get("texto", ""),
                score=h["_score"],
                metadata={"fonte": h["_source"].get("numero", h["_id"]),
                          "tipo": h["_source"].get("tipo", "")}
            )
            for h in resp["hits"]["hits"]
        ]
    
    def add_documents(self, documents: List[Dict]) -> None:
        for doc in documents:
            self.client.index(index=self.index, id=doc["id"], body=doc)
    
    def _simulado(self, query: str, top_k: int) -> List[Document]:
        """Resultados simulados quando OpenSearch não está disponível."""
        try:
            import json
            with open("../datasets/corpus_juridico_aula3.json", encoding="utf-8") as f:
                data = json.load(f)
            docs = data["documentos"][:top_k]
            return [Document(id=d["id"], content=d["texto"],
                            score=0.9 - i*0.05,
                            metadata={"fonte": d.get("numero", d["id"])})
                    for i, d in enumerate(docs)]
        except:
            return [Document(id=f"sim_{i}", content=f"Resultado simulado OpenSearch {i+1}",
                            score=0.9-i*0.1, metadata={}) for i in range(min(top_k, 3))]

os_retriever = OpenSearchRetriever(
    host=os.getenv("OPENSEARCH_HOST", "localhost"),
    port=int(os.getenv("OPENSEARCH_PORT", "9200")),
    password=os.getenv("OPENSEARCH_PASSWORD", "admin")
)

✅ OpenSearchRetriever conectado ao índice 'corpus_juridico'


## 🟢 Passo 4 — Implementar Retriever ChromaDB

In [8]:
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
import json

class ChromaDBRetriever(BaseRetriever):
    """Retriever usando ChromaDB com embeddings BGE-M3.
    
    MESMA INTERFACE que OpenSearchRetriever — intercambiável!
    """
    
    def __init__(self, collection_name: str = "corpus_juridico"):
        self._name = "ChromaDB"
        try:
            self.embedding_fn = SentenceTransformerEmbeddingFunction(
                model_name="BAAI/bge-m3",
                device="cuda" if __import__("torch").cuda.is_available() else "cpu"
            )
            self.chroma = chromadb.Client()
            self.collection = self.chroma.get_or_create_collection(
                name=collection_name,
                embedding_function=self.embedding_fn
            )
            self.connected = True
            print(f"✅ ChromaDBRetriever inicializado (coleção: {collection_name})")
            
            # Indexar corpus se coleção estiver vazia
            if self.collection.count() == 0:
                self._indexar_corpus()
        except Exception as e:
            self.connected = False
            print(f"⚠️  ChromaDB não disponível ({e}) — modo simulado")
    
    @property
    def name(self) -> str:
        return self._name
    
    def _indexar_corpus(self):
        """Indexa o corpus jurídico no ChromaDB."""
        try:
            with open("../datasets/corpus_juridico_aula3.json", encoding="utf-8") as f:
                data = json.load(f)
            docs = data["documentos"]
            
            self.collection.add(
                ids=[d["id"] for d in docs],
                documents=[d["texto"] for d in docs],
                metadatas=[{"fonte": d.get("numero", d["id"]), "tipo": d.get("tipo", "")} for d in docs]
            )
            print(f"   {len(docs)} documentos indexados no ChromaDB")
        except Exception as e:
            print(f"   Aviso: não foi possível indexar corpus ({e})")
    
    def search(self, query: str, top_k: int = 10) -> List[Document]:
        if not self.connected:
            return self._simulado(query, top_k)
        
        results = self.collection.query(
            query_texts=[query],
            n_results=min(top_k, max(self.collection.count(), 1))
        )
        
        docs = []
        for i, (doc_id, doc_text, distance, metadata) in enumerate(
            zip(results["ids"][0], results["documents"][0],
                results["distances"][0], results["metadatas"][0])
        ):
            score = 1.0 - distance  # ChromaDB retorna distância, converter para score
            docs.append(Document(id=doc_id, content=doc_text,
                                score=round(score, 4), metadata=metadata))
        return docs
    
    def add_documents(self, documents: List[Dict]) -> None:
        self.collection.add(
            ids=[d["id"] for d in documents],
            documents=[d.get("texto", d.get("content", "")) for d in documents],
            metadatas=[{"fonte": d.get("numero", d["id"])} for d in documents]
        )
    
    def _simulado(self, query: str, top_k: int) -> List[Document]:
        try:
            with open("../datasets/corpus_juridico_aula3.json", encoding="utf-8") as f:
                data = json.load(f)
            # ChromaDB usa similaridade vetorial — simular ordem diferente do OpenSearch
            docs = list(reversed(data["documentos"][:top_k]))
            return [Document(id=d["id"], content=d["texto"],
                            score=0.85 - i*0.05,
                            metadata={"fonte": d.get("numero", d["id"])})
                    for i, d in enumerate(docs)]
        except:
            return [Document(id=f"chroma_{i}", content=f"Resultado simulado ChromaDB {i+1}",
                            score=0.85-i*0.1, metadata={}) for i in range(min(top_k, 3))]

chroma_retriever = ChromaDBRetriever()

d:\IBMEC\MBA_RAG_CAG\.venv\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


✅ ChromaDBRetriever inicializado (coleção: corpus_juridico)
   80 documentos indexados no ChromaDB


## 🤖 Passo 5 — Implementar Reranker e Generator

In [9]:
from openai import OpenAI

class BGEReranker(BaseReranker):
    def __init__(self):
        import torch
        from transformers import AutoTokenizer, AutoModelForSequenceClassification
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        try:
            self.tok = AutoTokenizer.from_pretrained("BAAI/bge-reranker-v2-m3")
            self.model = AutoModelForSequenceClassification.from_pretrained(
                "BAAI/bge-reranker-v2-m3").eval().to(self.device)
            self.ok = True
        except:
            self.ok = False

    def rerank(self, query: str, docs: List[Document], top_k: int = 5) -> List[Document]:
        if not self.ok or not docs:
            return sorted(docs, key=lambda d: d.score, reverse=True)[:top_k]
        import torch
        pairs = [[query, d.content] for d in docs]
        inputs = self.tok(pairs, padding=True, truncation=True,
                         max_length=512, return_tensors="pt").to(self.device)
        with torch.no_grad():
            scores = torch.sigmoid(self.model(**inputs).logits.squeeze(-1)).cpu().numpy()
        for doc, s in zip(docs, scores):
            doc.score = round(float(s), 4)
        return sorted(docs, key=lambda d: d.score, reverse=True)[:top_k]


class LLMGenerator(BaseGenerator):
    """Generator com Groq primário (llama-3.1-8b-instant) + Ollama fallback."""

    def __init__(self):
        from dotenv import load_dotenv
        load_dotenv()
        groq_key   = os.getenv("GROQ_API_KEY", "")
        groq_model = os.getenv("GROQ_LLM_MODEL", "llama-3.1-8b-instant")
        ollama_url = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
        ollama_mod = os.getenv("OLLAMA_LLM_MODEL", "llama3.2:3b")

        self.client = None
        self.model = None
        self.provider = None

        if groq_key:
            try:
                c = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=groq_key)
                c.chat.completions.create(model=groq_model,
                                          messages=[{"role":"user","content":"ok"}],
                                          max_tokens=2, temperature=0.0)
                self.client, self.model, self.provider = c, groq_model, "groq"
            except Exception as e:
                print(f"⚠️  Groq falhou ({e}); caindo para Ollama…")

        if self.client is None:
            try:
                c = OpenAI(base_url=f"{ollama_url}/v1", api_key="ollama")
                c.chat.completions.create(model=ollama_mod,
                                          messages=[{"role":"user","content":"ok"}],
                                          max_tokens=2, temperature=0.0)
                self.client, self.model, self.provider = c, ollama_mod, "ollama"
            except Exception as e:
                print(f"❌ Nenhum LLM disponível para LLMGenerator: {e}")

    def generate(self, query: str, context: str) -> str:
        if self.client is None:
            return "[Geração não disponível]"
        prompt = f"""Você é assistente jurídico especializado em direito penal brasileiro.
Baseie-se EXCLUSIVAMENTE no contexto fornecido.

CONTEXTO:
{context}

PERGUNTA: {query}

RESPOSTA:"""
        try:
            r = self.client.chat.completions.create(
                model=self.model,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=400, temperature=0.1
            )
            return r.choices[0].message.content.strip()
        except Exception as e:
            return f"[Geração não disponível ({self.provider}): {e}]"


print("✅ BGEReranker e LLMGenerator (Groq + Ollama fallback) implementados")

✅ BGEReranker e LLMGenerator (Groq + Ollama fallback) implementados


## 🏗️ Passo 6 — Pipeline Modular RAG

In [10]:
class ModularRAGPipeline:
    """
    Pipeline RAG com módulos intercambiáveis.
    
    O pipeline só conhece as interfaces abstratas (BaseRetriever, etc.)
    NÃO conhece as implementações concretas (OpenSearch, ChromaDB, etc.)
    
    Isso permite trocar qualquer módulo sem alterar o pipeline.
    """
    
    def __init__(self,
                 retriever: BaseRetriever,
                 reranker: BaseReranker,
                 generator: BaseGenerator,
                 top_k_retrieval: int = 20,
                 top_k_final: int = 5):
        self.retriever = retriever
        self.reranker = reranker
        self.generator = generator
        self.top_k_retrieval = top_k_retrieval
        self.top_k_final = top_k_final
    
    def query(self, query: str) -> Dict:
        """
        Executa o pipeline completo:
        1. Retrieval → top_k_retrieval documentos
        2. Reranking → top_k_final documentos
        3. Geração → resposta final
        """
        import time
        
        # ETAPA 1: RETRIEVAL
        t0 = time.time()
        docs_retrieval = self.retriever.search(query, self.top_k_retrieval)
        t_retrieval = time.time() - t0
        
        # ETAPA 2: RERANKING
        t0 = time.time()
        docs_final = self.reranker.rerank(query, docs_retrieval, self.top_k_final)
        t_reranking = time.time() - t0
        
        # ETAPA 3: GERAÇÃO
        context = self._build_context(docs_final)
        t0 = time.time()
        response = self.generator.generate(query, context)
        t_generation = time.time() - t0
        
        return {
            "query": query,
            "retriever_usado": self.retriever.name,
            "docs_retrieved": len(docs_retrieval),
            "docs_final": [{"id": d.id, "score": d.score, "fonte": d.metadata.get("fonte", d.id)} 
                          for d in docs_final],
            "response": response,
            "latency": {
                "retrieval_ms": round(t_retrieval * 1000, 1),
                "reranking_ms": round(t_reranking * 1000, 1),
                "generation_ms": round(t_generation * 1000, 1),
                "total_ms": round((t_retrieval + t_reranking + t_generation) * 1000, 1)
            }
        }
    
    def _build_context(self, docs: List[Document]) -> str:
        return "\n\n".join([
            f"[{i+1}] {d.metadata.get('fonte', d.id)} (score: {d.score:.3f}):\n{d.content[:400]}..."
            for i, d in enumerate(docs)
        ])
    
    def swap_retriever(self, new_retriever: BaseRetriever) -> None:
        """Troca o retriever SEM alterar nenhum outro módulo."""
        old_name = self.retriever.name
        self.retriever = new_retriever
        print(f"🔄 Retriever trocado: {old_name} → {new_retriever.name}")

print("✅ ModularRAGPipeline definido")
print()
print("Módulos disponíveis:")
print("  Retrievers:  OpenSearchRetriever, ChromaDBRetriever")
print("  Rerankers:   BGEReranker")
print("  Generators:  LLMGenerator (Groq + Ollama fallback)")

✅ ModularRAGPipeline definido

Módulos disponíveis:
  Retrievers:  OpenSearchRetriever, ChromaDBRetriever
  Rerankers:   BGEReranker
  Generators:  LLMGenerator (Groq + Ollama fallback)


## 🧪 Passo 7 — Demonstrar Intercambialidade

In [11]:
import time

reranker  = BGEReranker()
generator = LLMGenerator()

query_teste = "O suspeito é obrigado a falar na delegacia?"

print("=" * 65)
print("🔬 DEMONSTRAÇÃO DE INTERCAMBIALIDADE DO RETRIEVER")
print("=" * 65)
print(f"Query: '{query_teste}'")
print()

# PIPELINE COM OPENSEARCH
pipeline_os = ModularRAGPipeline(
    retriever=os_retriever,
    reranker=reranker,
    generator=generator
)
print("⏳ Executando com OpenSearch...")
result_os = pipeline_os.query(query_teste)

print(f"✅ OpenSearch concluído em {result_os['latency']['total_ms']}ms")
print()

# TROCAR RETRIEVER — NENHUMA OUTRA LINHA DO PIPELINE MUDA!
pipeline_os.swap_retriever(chroma_retriever)

print("⏳ Executando com ChromaDB (mesmo pipeline, retriever trocado)...")
result_chroma = pipeline_os.query(query_teste)

print(f"✅ ChromaDB concluído em {result_chroma['latency']['total_ms']}ms")
print()

# COMPARAR RESULTADOS
print("─" * 65)
print("📊 COMPARAÇÃO DOS RESULTADOS")
print("─" * 65)
print()
print("🔵 OpenSearch:")
print(f"   Docs recuperados: {result_os['docs_retrieved']}")
print(f"   Resposta: {result_os['response'][:200]}...")
print()
print("🟢 ChromaDB:")
print(f"   Docs recuperados: {result_chroma['docs_retrieved']}")
print(f"   Resposta: {result_chroma['response'][:200]}...")
print()

# Verificar overlap dos docs finais
ids_os = {d["id"] for d in result_os["docs_final"]}
ids_chroma = {d["id"] for d in result_chroma["docs_final"]}
overlap = len(ids_os & ids_chroma)
print(f"📌 Documentos em comum no top-5: {overlap}/5")
print(f"   Overlap: {overlap/5:.0%}")

🔬 DEMONSTRAÇÃO DE INTERCAMBIALIDADE DO RETRIEVER
Query: 'O suspeito é obrigado a falar na delegacia?'

⏳ Executando com OpenSearch...
✅ OpenSearch concluído em 8125.6ms

🔄 Retriever trocado: OpenSearch → ChromaDB
⏳ Executando com ChromaDB (mesmo pipeline, retriever trocado)...
✅ ChromaDB concluído em 8232.7ms

─────────────────────────────────────────────────────────────────
📊 COMPARAÇÃO DOS RESULTADOS
─────────────────────────────────────────────────────────────────

🔵 OpenSearch:
   Docs recuperados: 20
   Resposta: Não, o suspeito não é obrigado a falar na delegacia. O direito ao silêncio, garantido pelo art. 5º, LXIII, da Constituição Federal, deve ser observado no interrogatório policial e judicial. O investig...

🟢 ChromaDB:
   Docs recuperados: 20
   Resposta: Não, o suspeito não é obrigado a falar na delegacia. O direito ao silêncio é garantido constitucionalmente pelo art. 5º, LXIII, da Constituição Federal e deve ser observado no interrogatório policial ...

📌 Documentos em c

## ✅ Passo 8 — Análise e Conclusão

In [12]:
print("📝 ANÁLISE DE MODULARIDADE")
print("=" * 55)
print()
print("✅ O que demonstramos:")
print("   1. OpenSearchRetriever e ChromaDBRetriever implementam BaseRetriever")
print("   2. ModularRAGPipeline só conhece BaseRetriever — não a implementação")
print("   3. swap_retriever() trocou o módulo sem alterar reranker ou generator")
print("   4. Os resultados podem diferir (diferentes algoritmos de busca)")
print()
print("💡 Princípio aplicado: Inversão de Dependência (SOLID)")
print("   'Dependa de abstrações, não de implementações concretas'")
print()
print("🔧 Extensões possíveis sem alterar o pipeline:")
print("   - Adicionar FAISSRetriever(BaseRetriever)")
print("   - Adicionar LLMReranker(BaseReranker)")
print("   - Adicionar OpenAIGenerator(BaseGenerator)")
print()
print("✅ LAB 4 CONCLUÍDO!")
print()
print("📋 CHECKLIST DE ENTREGA:")
print("   [ ] Interfaces abstratas implementadas")
print("   [ ] OpenSearchRetriever e ChromaDBRetriever funcionais")
print("   [ ] Pipeline Modular RAG executou com ambos os retrievers")
print("   [ ] swap_retriever() demonstrado")
print("   [ ] Análise de overlap dos resultados escrita")

📝 ANÁLISE DE MODULARIDADE

✅ O que demonstramos:
   1. OpenSearchRetriever e ChromaDBRetriever implementam BaseRetriever
   2. ModularRAGPipeline só conhece BaseRetriever — não a implementação
   3. swap_retriever() trocou o módulo sem alterar reranker ou generator
   4. Os resultados podem diferir (diferentes algoritmos de busca)

💡 Princípio aplicado: Inversão de Dependência (SOLID)
   'Dependa de abstrações, não de implementações concretas'

🔧 Extensões possíveis sem alterar o pipeline:
   - Adicionar FAISSRetriever(BaseRetriever)
   - Adicionar LLMReranker(BaseReranker)
   - Adicionar OpenAIGenerator(BaseGenerator)

✅ LAB 4 CONCLUÍDO!

📋 CHECKLIST DE ENTREGA:
   [ ] Interfaces abstratas implementadas
   [ ] OpenSearchRetriever e ChromaDBRetriever funcionais
   [ ] Pipeline Modular RAG executou com ambos os retrievers
   [ ] swap_retriever() demonstrado
   [ ] Análise de overlap dos resultados escrita
